# RAG Knowledge Base

Eagle, aviary, birds, ornithological

Resources: [Philippine Eagle Foundation](https://www.philippineeaglefoundation.org/resources)
1. [PDF Publication] [Priority conservation areas and a global population estimate for the critically endangered Philippine Eagle](https://www.philippineeaglefoundation.org/_files/ugd/d08c94_74d2622c339e4f1499c76d07852a85a7.pdf)
2. [Philippine Eagle “Sinabadan” Rescued at Mt Tangkulan](https://www.philippineeaglefoundation.org/_files/ugd/d08c94_9e2dfe5081234dfc81ad12c5ab1421f2.pdf)
3. [Preventing Philippine Eagle hunting: what are we missing?](https://www.threatenedtaxa.org/index.php/JoTT/article/view/2301/3831)
4. [First Record of a Successful Nesting of South Philippine Hawk-Eagles
(Nisaetus pinskeri) on Mindanao, Philippines](https://www.researchgate.net/profile/Tristan-Luap-Senarillos/publication/399615160_First_Record_of_a_Successful_Nesting_of_South_Philippine_Hawk-Eagles_Nisaetus_pinskeri_on_Mindanao_Philippines/links/69612b090f6f9e478e436913/First-Record-of-a-Successful-Nesting-of-South-Philippine-Hawk-Eagles-Nisaetus-pinskeri-on-Mindanao-Philippines.pdf)
5. [Population Viability Analysis to Inform
Conservation Planning for the Philippine Eagle](https://www.cpsg.org/sites/default/files/2025-12/Valle%20et%20al.%202025%20_%20Population%20Viability%20Analysis%20to%20Inform%20Conservation%20Planning%20for%20the%20Philippine%20Eagle%20%28Pithecophaga%20jefferyi%29.pdf)
6. [Genomic analysis reveals recent population
decline and exceptionally low genome-wide
heterozygosity of the critically endangered
Philippine eagle, Pithecophaga jefferyi (Aves:
Accipitridae)](https://link.springer.com/article/10.1186/s12864-026-12859-9)

News articles:
1. [The Oven of Life](https://www.gmanetwork.com/news/specials/andypenafuerte/284/the-oven-of-life/)
2. [Saving the endangered Philippine eagle from extinction](https://mb.com.ph/2026/06/10/saving-the-endangered-philippine-eagle-from-extinction)
3. [Harmonizing Policy and Action Through the Philippine Eagle Population Viability Analysis and Action Plan Updating](https://faps.bmb.gov.ph/faps/2025/09/29/harmonizing-policy-and-action-through-the-philippine-eagle-population-viability-analysis-and-action-plan-updating/)

* Philippine Eagle Foundation
* Biodiversity Management Bureau Gov PH
    * [Facts & Figures - Wildlife](https://www.bmb.gov.ph/protection-and-conservation-of-wildlife/facts-figures-wildlife/)
* BirdLife
* 

In [1]:
import os
import re
from collections import Counter

import bs4
import requests

from datasets import Dataset

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_core.documents import Document

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import START, StateGraph

from openai import OpenAI

from typing_extensions import List, TypedDict

# from ragas import evaluate
# from ragas.metrics import faithfulness

/home/linuxg/Desktop/Turing/animal-conservation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_41027/1297568184.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
pdf_folder = "research"

docs = []

for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder, file))
        pdf_docs = loader.load()

        for d in pdf_docs:
            d.metadata["source_type"] = "pdf"
            d.metadata["source"] = file

        docs.extend(pdf_docs)

In [3]:
urls = [
    "https://www.philippineeaglefoundation.org/philippine-eagle",
    "https://datazone.birdlife.org/species/factsheet/philippine-eagle-pithecophaga-jefferyi",
    "https://www.gmanetwork.com/news/specials/andypenafuerte/284/the-oven-of-life/",
    "https://mb.com.ph/2026/06/10/saving-the-endangered-philippine-eagle-from-extinction",
]

In [4]:
for url in urls:
    loader = WebBaseLoader(web_paths=(url,))
    web_docs = loader.load()

    for d in web_docs:
        d.metadata["source_type"] = "web"
        d.metadata["source_url"] = url

    docs.extend(web_docs)

In [5]:
line_counts = Counter()

for doc in docs:
    for line in doc.page_content.split("\n"):
        line = line.strip()
        if line:
            line_counts[line] += 1

noise_lines = {line for line, count in line_counts.items() if count >= 10}


def remove_repeated_lines(text):
    return "\n".join(
        line for line in text.split("\n") if line.strip() not in noise_lines
    )


def clean_text(text):
    patterns = [
        r"\{\{.*?\}\}",
        r"@media\s*\([^\)]*\)",
        r"\.[A-Za-z0-9_-]+\s*\{",
        r"padding-[^;]+;",
        r"font-size[^;]+;",
        r"margin-[^;]+;",
        r"return;",
        r"Downloaded from https?://.*",
        r"doi:\s*\S+",
        r"Journal of .*?(?=\n|$)",
    ]

    for pattern in patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)

    text = remove_emails(text)

    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)

    return text.strip()


EMAIL_PATTERN = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"


def remove_emails(text):
    return re.sub(EMAIL_PATTERN, "[EMAIL REDACTED]", text)


def remove_pdf_headers(text):
    lines = text.split("\n")

    cleaned = []
    for line in lines:
        if (
            "KEY WORDS" in line
            or "doi:" in line.lower()
            or line.strip().startswith("/")
            or "Raptor Research Foundation" in line
        ):
            continue
        cleaned.append(line)

    return "\n".join(cleaned)


def remove_pdf_metadata(text):
    return re.sub(r"\{.*?\}", "", text, flags=re.DOTALL)


def fix_broken_words(text):
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

In [6]:
cleaned_docs = []

for doc in docs:
    text = doc.page_content

    text = fix_broken_words(text)
    text = remove_repeated_lines(text)
    text = remove_pdf_headers(text)
    text = remove_pdf_metadata(text)
    text = clean_text(text)

    doc.page_content = text
    cleaned_docs.append(doc)

docs = cleaned_docs

print("Total documents loaded:", len(docs))

Total documents loaded: 117


## Splitting

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=300)

chunks = splitter.split_documents(docs)

print(len(chunks))

327


In [8]:
chunks = [
    chunk
    for chunk in chunks
    if not (
        ("@media" in chunk.page_content)
        or ("{{" in chunk.page_content)
        or ("padding-bottom" in chunk.page_content)
        or ("font-size" in chunk.page_content)
    )
]

print(len(chunks))

327


Adding metadata to chunks

In [9]:
for chunk in chunks:
    text = chunk.page_content.lower()

    if "method" in text:
        chunk.metadata["section"] = "methods"
    elif "result" in text or "findings" in text or "observed" in text:
        chunk.metadata["section"] = "results"
    elif "introduction" in text or "background" in text or "overview" in text:
        chunk.metadata["section"] = "intro"
    else:
        chunk.metadata["section"] = "other"

In [10]:
for i, chunk in enumerate(chunks[:30]):
    print(f"\n--- CHUNK {i} ---")
    print(chunk.page_content)
    print(chunk.metadata)


--- CHUNK 0 ---
From these platforms, we heard the food-begging calls of an immature South Philippine Hawk-Eagle, and at the same time, we saw an adult South Philippine Hawk-Eagle soaring over a nearby ridge multiple times. We then observed an adult South Philippine Hawkeagle carrying a sprig into a densely forested slope, suggesting potential nest-building activity. Investigating the area on 21 July 2023, we found a South Philippine Hawk-Eagle nest containing one nestling approximately 70 m from one of our platforms (Fig. 1). When adults visited the nest, we distinguished the male from the female by size and by distinctive plumage: the male had darker breast and facial plumage than the female. Behavioral cues further supported identification, as the female was observed feeding the young, while the male was not seen doing so, and instead appeared limited to the role of delivering prey to the nest. When we found the nest, the nestling appeared bulkier than the adults, with dense plumag

In [11]:
for i in range(len(chunks) - 1):
    print(i, chunks[i].page_content[-100:])

0 situated in a riparian forest at approximately 1,200 masl within Mt. Apo Natural Park on the Balabag
1 roximity to small-scale clearings and regenerating farmland underscored its persistence in partially
2 wing the nestling standing while the adult female perches on a nearby branch. Photo by R. L. Taraya.
3 h prey item and its relative contribution to the total prey biomass as (individual biomass /C4 total
4 et. Throughout the observation period, we observed the female more frequently near the nest than the
5 fecating posture; (B) wing-flapping exercise; (C) standing position. Photo by R. L. Taraya. Letter 3
6 le 2). Mammals made up 70 % (n ¼ 16) of all prey items. The second most common prey class was birds,
7 ivering food, while the female remained and fed the nestling. She was observed plucking the feathers
8 8.7) 0.40 0.80 22.1 Unidentified bats 3 (13) UNK UNK UNK Reptiles Green crested lizard ( Bronchocela
9 023 Flying to trees approximately 70 –80 m from the nest with loud voca

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16821.64it/s]


In [13]:
vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory="chroma_db"
)

Similarity search testing

In [14]:
query = "What are the 2026 conservation efforts for Philippine eagles?"

query_vector = embeddings.embed_query(query)

results = vectorstore.similarity_search_by_vector(query_vector, k=3)

for i, doc in enumerate(results):
    print(f"\n--- RESULT {i} ---")
    print(doc.page_content[:500])
    print(doc.metadata)


--- RESULT 0 ---
his family.This behavioral barrier is echoed by Mariglo Rosaida Laririt, Assistant Director at the Biodiversity Management Bureau (BMB) of the Department of Environment and Natural Resources (DENR).“Wildlife persecution is very much a part of our daily life in the Philippines,” Laririt observes, noting that threats range from children using slingshots to adults using unregulated air guns. DENR Biodiversity Management Bureau Asst. Dir. Mariglo Laririt (second to the right) was among the Filipino 
{'source': 'https://www.gmanetwork.com/news/specials/andypenafuerte/284/the-oven-of-life/', 'source_type': 'web', 'source_url': 'https://www.gmanetwork.com/news/specials/andypenafuerte/284/the-oven-of-life/', 'section': 'other', 'description': 'Inside the high-tech and high-heart mission to save the Philippine eagle.', 'language': 'en', 'title': 'The Oven of Life | Cover Stories | GMA News Online'}

--- RESULT 1 ---
the Philippine Eagle Species Action Plan (PESAP) 2024-2030 in